# 03 — Symmetries and shifts

Notebook 02 built a model that predicts cosmological parameters from a catalog and scored it
on held-out simulations. That is the ordinary machine-learning setup, and it is not what you
are scored on here.

Here the held-out catalogs are **changed** before your model sees them. Some of the changes
do not alter the physics at all. Some genuinely destroy information. One of them is a whole
different simulation code. You are scored separately under each, so the leaderboard is a
table rather than a number, and a model can win a column it is built for while losing another.

This notebook is about the first kind, because it is the one where the answer is knowable in
advance:

> If a change does not alter the physics, a correctly built model must give **the same
> answer**. If it does not, you have found a defect in your model — not a limit of the data.

By the end you will have:

1. seen exactly what each condition does to a catalog file,
2. watched three representations of the same catalog react completely differently to the
   same three symmetries,
3. seen which property of a representation buys which invariance, and
4. measured what the corruptions actually cost.

In [ ]:
# Setup. On Colab this installs the toolkit, mounts the data bucket and points the
# environment variables at it. Anywhere else -- a cluster with the data already on disk --
# it does nothing, which is why there is one set of notebooks rather than two.
import sys

if "google.colab" in sys.modules:
    # --force-reinstall, every time, on purpose. Installing only when the package is
    # missing means anyone who ran a notebook once keeps a stale copy forever, and during
    # an event where fixes are being pushed that is exactly backwards. --no-deps keeps it
    # to a few seconds: everything it depends on is already in the runtime.
    %pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/xwzhang98/kaai-robust-inference-hackathon-2026
    # Drop anything already imported from the old copy, so this works without a restart.
    for _name in [m for m in sys.modules if m.startswith("kaai_hackathon")]:
        del sys.modules[_name]

from kaai_hackathon.colab_setup import setup

setup()

## Setup

In [ ]:
%matplotlib inline
import os
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from kaai_hackathon import PUBLIC_SUITES
from kaai_hackathon.catalog_io import read_catalog, validate_linkage
from kaai_hackathon.conditions import (
    PUBLISHED_CONDITIONS, apply_condition, condition, condition_seed,
)
from kaai_hackathon.features import catalog_features
from kaai_hackathon.progress import track
from kaai_hackathon.splits import example_sims, load_labels, local_split

DATA_ROOT = Path(os.environ["CAMELS_HACKATHON_DATA"])
PARAMS_ROOT = Path(os.environ.get("CAMELS_HACKATHON_PARAMS", DATA_ROOT))

# Enough columns to apply every published condition faithfully. A shift that moves
# positions has to move GroupPos and the centres of mass too, not just SubhaloPos.
GROUP_FIELDS = ["GroupPos", "GroupCM", "GroupVel", "GroupMass",
                "GroupNsubs", "GroupFirstSub"]
SUBHALO_FIELDS = ["SubhaloPos", "SubhaloCM", "SubhaloVel", "SubhaloSpin",
                  "SubhaloMass", "SubhaloMassType", "SubhaloGrNr", "SubhaloParent"]


def load(suite, sim_id):
    return read_catalog(DATA_ROOT / suite / f"LH_{sim_id}" / "groups_090.hdf5",
                        group_fields=GROUP_FIELDS, subhalo_fields=SUBHALO_FIELDS)

# A simulation that is certain to exist. The public ids are a pinned random 900
# out of 1000, not 0..899, so `LH_2` may simply not be in the data you were given.
EXAMPLE = example_sims("IllustrisTNG")[0]



cat = load("IllustrisTNG", EXAMPLE)
print(f"{cat.n_groups} halos, {cat.n_subhalos} subhalos, box {cat.box_size:.0f} ckpc/h")

## 1. The conditions

They are not free-form. Each one is a named entry in a registry that the **scorer itself
uses**, so what you apply here and what the organizers apply at scoring time cannot drift
apart.

In [ ]:
print(f"{'name':16s} {'kind':12s} {'operation':22s} settings")
KIND = {"clean": "unchanged", "S": "symmetry", "C": "corruption"}
for spec in PUBLISHED_CONDITIONS:
    print(f"{spec.name:16s} {KIND[spec.tier]:12s} {spec.op:22s} {spec.params or ''}")

Two practical points about how these get applied.

**The seeds are pinned.** A condition's randomness comes from a hash of
`(base_seed, condition, suite, sim_id)`, so every team is scored on byte-identical inputs and
nobody is unlucky. The organizers' base seed is not published, so you cannot precompute the
exact test files — but you can reproduce this notebook exactly.

**Shifted catalogs are generated, not stored.** With pinned seeds there is no reason to keep
them on disk.

In [ ]:
print("same base seed, same catalog  ->",
      condition_seed(7, "pos_noise_lo", "SIMBA", 42),
      condition_seed(7, "pos_noise_lo", "SIMBA", 42))
print("different simulation          ->", condition_seed(7, "pos_noise_lo", "SIMBA", 43))
print("different condition           ->", condition_seed(7, "mass_noise", "SIMBA", 42))

## 2. What a symmetry actually does to the file

Start with the one that is easiest to see. `translate` adds the same random vector to every
position and wraps at the box edge.

In [ ]:
MSTAR_CUT = 1.3e-2          # 1e10 Msun/h units


def galaxies(c, cut=MSTAR_CUT):
    keep = c.subhalo["SubhaloMassType"][:, 4] > cut
    return np.mod(c.subhalo["SubhaloPos"][keep], c.box_size) / 1000.0     # -> cMpc/h


shifted = {name: apply_condition(cat, condition(name),
                                 seed=condition_seed(7, name, "IllustrisTNG", 0))
           for name in ("clean", "translate", "rotate90", "permute")}

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for ax, name in zip(axes, ("clean", "translate", "rotate90")):
    g = galaxies(shifted[name])
    ax.scatter(g[:, 0], g[:, 1], s=5, alpha=0.6)
    ax.set_xlim(0, 25); ax.set_ylim(0, 25); ax.set_aspect("equal")
    ax.set_title(name); ax.set_xlabel("x [cMpc/h]")
axes[0].set_ylabel("y [cMpc/h]")
plt.tight_layout()

Every galaxy is somewhere else. Structures that ran off one edge now come back at the other.
And yet nothing about the *physics* has changed — there is no origin in a periodic box, and
no preferred orientation.

The way to see that is to measure something that does not depend on where you put the origin.
The distribution of pair separations is such a thing.

In [ ]:
from scipy.spatial import cKDTree

RADII = np.linspace(0.0, 5.0, 26)          # cMpc/h


def pair_counts(c, cut=MSTAR_CUT):
    g = galaxies(c, cut)
    tree = cKDTree(np.mod(g, 25.0), boxsize=25.0)
    return np.asarray(tree.count_neighbors(tree, RADII[1:]), dtype=np.float64)


reference = pair_counts(shifted["clean"])
print(f"{'condition':12s} {'galaxies':>9s} {'largest relative change in pair counts':>40s}")
for name in ("clean", "translate", "rotate90", "permute"):
    counts = pair_counts(shifted[name])
    change = np.max(np.abs(counts - reference) / np.maximum(reference, 1))
    print(f"{name:12s} {len(galaxies(shifted[name])):9d} {change:40.2e}")

Identical, to floating-point rounding. Same galaxies, same clustering, same everything a
physical model should care about.

### The details that make it a real symmetry

**All seven vector fields rotate together.** `GroupPos`, `GroupCM`, `GroupVel`, `SubhaloPos`,
`SubhaloCM`, `SubhaloVel` and `SubhaloSpin`. Rotating positions while leaving velocities
pointing the old way would give a catalog no simulation could produce, and then the condition
would be measuring our bug rather than your robustness.

In [ ]:
turned = shifted["rotate90"]
centre = np.full(3, cat.box_size / 2.0)

for table, names in (("group", ("GroupPos", "GroupCM", "GroupVel")),
                     ("subhalo", ("SubhaloPos", "SubhaloCM", "SubhaloVel", "SubhaloSpin"))):
    store_before = cat.group if table == "group" else cat.subhalo
    store_after = turned.group if table == "group" else turned.subhalo
    for name in names:
        before = store_before[name].astype(np.float64)
        after = store_after[name].astype(np.float64)
        # Positions turn about the CENTRE of the box, so what a rotation preserves is the
        # distance to the centre. Velocities and spin are free vectors and turn about zero,
        # so for them it is the plain length.
        if "Pos" in name or name.endswith("CM"):
            before, after = before - centre, after - centre
        kept = np.allclose(np.linalg.norm(before, axis=-1),
                           np.linalg.norm(after, axis=-1), atol=1e-2)
        moved = not np.allclose(store_before[name], store_after[name])
        print(f"{name:16s} values changed={str(moved):5s}   length preserved={kept}")

**Only 24 rotations are allowed.** An arbitrary 3D rotation does not preserve a periodic
cube: the rotated box stops tiling space, and the structure at the boundary is destroyed.
Only the rotations that map the cube onto itself keep the periodicity exact — there are 24 of
them with determinant $+1$.

Reflections are excluded on purpose. `SubhaloSpin` is a pseudo-vector and would need an extra
sign flip under a reflection; keeping to proper rotations removes the subtlety entirely.

In [ ]:
from kaai_hackathon.shifts import cubic_rotations

R = cubic_rotations()
print(f"{len(R)} matrices")
print(f"all have det = +1:            {np.allclose(np.linalg.det(R), 1.0)}")
print(f"all are orthogonal:           "
      f"{np.allclose(R @ np.transpose(R, (0, 2, 1)), np.eye(3))}")
print(f"all entries are 0, +1 or -1:  {np.all(np.isin(R, (-1, 0, 1)))}")
print("\none of them:\n", R[7].astype(int))

**`permute` shuffles rows within each halo.** A Subfind file keeps `SubhaloGrNr` sorted and
defines `GroupFirstSub` and `SubhaloParent` against that ordering, so a global shuffle would
produce a file no Subfind run could emit. Shuffling *within* each halo removes the row order a
model might lean on and leaves a catalog that is still internally consistent — which the
scorer checks.

In [ ]:
validate_linkage(shifted["permute"])
print("linkage still self-consistent after permutation")

grnr_before = cat.subhalo["SubhaloGrNr"]
grnr_after = shifted["permute"].subhalo["SubhaloGrNr"]
print(f"SubhaloGrNr unchanged:        {np.array_equal(grnr_before, grnr_after)}")

mass_before = cat.subhalo["SubhaloMass"]
mass_after = shifted["permute"].subhalo["SubhaloMass"]
print(f"rows that now hold a different subhalo: "
      f"{int((mass_before != mass_after).sum())} of {len(mass_before)}")
print(f"the multiset of masses is the same:     "
      f"{np.array_equal(np.sort(mass_before), np.sort(mass_after))}")

## 3. The point of the whole exercise

Three ways to turn a catalog into a fixed-length vector. All three are things people
genuinely build. They differ only in what they choose to be blind to.

| representation | what it is | blind to |
|---|---|---|
| `raw_rows` | the first 64 subhalo rows **as they appear in the file**: $(x, y, z, \log M_\star)$ each, flattened | nothing |
| `sorted_rows` | the 64 most massive galaxies, sorted by $M_\star$: the same four numbers each | row order |
| `density_grid` | galaxy counts in an $8\times8\times8$ grid over the box | row order |
| `summary` | counts, a stellar-mass histogram, periodic neighbour counts | row order, where the origin is, which way is up |

`raw_rows` is what you write first. `sorted_rows` is what you write once you notice that row
order is arbitrary. `density_grid` is what you write when you want to use a CNN — it is the
standard way to hand a point cloud to a convolutional network, and unlike the two row-based
ones it actually reads the spatial structure. `summary` is notebook 02's feature vector.

Note where the grid sits: it has an explicit origin (cell 0 starts at $x=0$) and an explicit
orientation (the axes). Keep that in mind when you read the table.

Fit the same Ridge regression on all four, then score each under the three symmetries.

In [ ]:
K_GALAXIES = 64


def _top_positions_and_masses(c, cut=MSTAR_CUT):
    mstar = c.subhalo["SubhaloMassType"][:, 4]
    keep = mstar > cut
    pos = np.mod(c.subhalo["SubhaloPos"][keep], c.box_size) / c.box_size
    return pos, mstar[keep]


def raw_rows(c):
    # rows exactly as the file orders them -- no sorting anywhere
    pos, mstar = _top_positions_and_masses(c)
    block = np.zeros((K_GALAXIES, 4), dtype=np.float64)
    n = min(K_GALAXIES, len(pos))
    block[:n, :3] = pos[:n]
    block[:n, 3] = np.log10(np.maximum(mstar[:n], 1e-8))
    return block.ravel()


def sorted_rows(c):
    pos, mstar = _top_positions_and_masses(c)
    order = np.argsort(mstar)[::-1][:K_GALAXIES]
    block = np.zeros((K_GALAXIES, 4), dtype=np.float64)
    block[:len(order), :3] = pos[order]
    block[:len(order), 3] = np.log10(np.maximum(mstar[order], 1e-8))
    return block.ravel()


GRID = 8


def density_grid(c):
    pos, _ = _top_positions_and_masses(c)
    cell = np.minimum((pos * GRID).astype(int), GRID - 1)
    counts = np.zeros((GRID, GRID, GRID))
    np.add.at(counts, (cell[:, 0], cell[:, 1], cell[:, 2]), 1.0)
    return np.log1p(counts).ravel()


REPRESENTATIONS = {"raw_rows": raw_rows,
                   "sorted_rows": sorted_rows,
                   "density_grid": density_grid,
                   "summary": catalog_features}

In [ ]:
N_TRAIN_PER_SUITE = 250
N_TEST_PER_SUITE = 60
BASE_SEED = 7
SYMMETRIES = ("clean", "translate", "rotate90", "permute")

started = time.time()
train_x = {name: [] for name in REPRESENTATIONS}
train_y = []
for suite in PUBLIC_SUITES:
    labels = load_labels(PARAMS_ROOT, suite)
    for sim_id in track(local_split(suite)["train"][:N_TRAIN_PER_SUITE],
                        f"{suite}: training set"):
        c = load(suite, sim_id)
        for name, fn in REPRESENTATIONS.items():
            train_x[name].append(fn(c))
        train_y.append(labels[sim_id, :2])
train_y = np.asarray(train_y)
print(f"{len(train_y)} training catalogs in {time.time() - started:.0f}s")

In [ ]:
started = time.time()
test_x = {(name, cond): [] for name in REPRESENTATIONS for cond in SYMMETRIES}
test_y = []
for suite in PUBLIC_SUITES:
    labels = load_labels(PARAMS_ROOT, suite)
    for sim_id in track(local_split(suite)["test"][:N_TEST_PER_SUITE],
                        f"{suite}: symmetries"):
        c = load(suite, sim_id)
        for cond in SYMMETRIES:
            spec = condition(cond)
            moved = apply_condition(c, spec, condition_seed(BASE_SEED, cond, suite, sim_id))
            for name, fn in REPRESENTATIONS.items():
                test_x[(name, cond)].append(fn(moved))
        test_y.append(labels[sim_id, :2])
test_y = np.asarray(test_y)
print(f"{len(test_y)} test catalogs x {len(SYMMETRIES)} conditions "
      f"in {time.time() - started:.0f}s")

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

from kaai_hackathon.scoring import r2

TARGETS = ("Omega_m", "sigma_8")
fits = {}
for name in REPRESENTATIONS:
    x = np.asarray(train_x[name])
    scaler = StandardScaler().fit(x)
    fits[name] = (scaler, RidgeCV(alphas=np.logspace(-3, 4, 30))
                  .fit(scaler.transform(x), train_y))

scores = {}
for name in REPRESENTATIONS:
    scaler, model = fits[name]
    for cond in SYMMETRIES:
        prediction = model.predict(scaler.transform(np.asarray(test_x[(name, cond)])))
        scores[(name, cond)] = [r2(test_y[:, j], prediction[:, j]) for j in range(2)]

for j, target in enumerate(TARGETS):
    print(f"\n### {target}   (R^2, and the change from `clean`)")
    print(f"| {'representation':14s} | " +
          " | ".join(f"{c:^16s}" for c in SYMMETRIES) + " |")
    print("|" + "-" * 16 + "|" + "|".join(["-" * 18] * len(SYMMETRIES)) + "|")
    for name in REPRESENTATIONS:
        base = scores[(name, "clean")][j]
        cells = []
        for cond in SYMMETRIES:
            value = scores[(name, cond)][j]
            cells.append(f"{value:+.3f}" if cond == "clean"
                         else f"{value:+.3f} ({value - base:+.3f})")
        print(f"| {name:14s} | " + " | ".join(f"{c:^16s}" for c in cells) + " |")

Read that table one row at a time, and notice that it does **not** say what you probably
expected it to say.

**`permute` is the one that bites.** `raw_rows` reads slot 1 as "whatever subhalo happens to
be written first", so shuffling the rows hands the model a different galaxy in every slot.
That is a real loss and a large one, especially on $\sigma_8$. `sorted_rows` is the same
representation with one line added — sort by stellar mass — and that line takes the condition
to exactly zero, because sorting makes the vector a function of the *set* of galaxies rather
than of the order someone wrote them down in.

**`summary` does not move under any of the three**, exactly, to the last digit. Counts, a
mass histogram and periodic neighbour counts are functions of quantities that have no origin
and no orientation. The invariance is a property of the representation, so it holds for every
catalog without training on a single shifted example.

**And now the part that is worth sitting with: `translate` and `rotate90` barely move
anything here** — not even `density_grid`, which has an explicit origin and explicit axes and
is the one representation in the table that genuinely reads spatial structure.

That is not a bug in the experiment. It is a fact about the problem:

> Absolute position and absolute orientation carry **no cosmological information**. So a
> model fitted on this data has already learned to ignore them. You cannot be hurt by losing
> something you never used.

Which means these two conditions are testing something narrower than "does your model handle
a shifted box". They test whether your model has an **internal reference point or a preferred
direction that it actually relies on**. A histogram has neither. A grid has both, but what it
learned from them was the count distribution, which survives.

The models that do get hurt are the ones sophisticated enough to build geometry out of
positions — and there the failure is subtle, easy to miss, and published. Notebook 04 takes
the same instruments to a graph neural network and finds two of them, one costing
$\sigma_8$ about 0.05.

The general lesson survives all of this intact: **you do not have to learn a symmetry if you
build it in.** Sorting bought one condition for one line. Augmenting with shifted copies would
get you part of the way at the cost of data and training time, and it would still only be
approximate.

## 4. The corruptions

Different question entirely. These really do remove information, so losing accuracy is not a
defect. What is being measured is how much.

Look at what they do first.

In [ ]:
CORRUPTIONS = ("pos_noise_lo", "pos_noise_hi", "vel_noise", "mass_noise")

corrupted = {name: apply_condition(cat, condition(name),
                                   seed=condition_seed(BASE_SEED, name, "IllustrisTNG", 0))
             for name in CORRUPTIONS}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

ax = axes[0]
reference = pair_counts(cat)
for name in ("pos_noise_lo", "pos_noise_hi"):
    ax.plot(RADII[1:], pair_counts(corrupted[name]) / reference, marker="o", ms=3.5,
            label=f"{name} (sigma = {condition(name).params['sigma_ckpch']:.0f} ckpc/h)")
ax.axhline(1.0, color="0.4", ls="--", lw=1)
ax.set_xlabel("separation [cMpc/h]"); ax.set_ylabel("pair counts / clean")
ax.set_title("position noise smears out clustering"); ax.legend()

ax = axes[1]
bins = np.linspace(-2.5, 2.0, 40)
for name, c in (("clean", cat), ("mass_noise", corrupted["mass_noise"])):
    m = c.subhalo["SubhaloMassType"][:, 4]
    ax.hist(np.log10(m[m > 1e-4]), bins=bins, histtype="step", lw=1.6, label=name)
ax.set_yscale("log")
ax.set_xlabel(r"$\log_{10}(M_\star\,/\,10^{10}M_\odot/h)$"); ax.set_ylabel("galaxies")
ax.set_title(f"mass noise, sigma = {condition('mass_noise').params['sigma_dex']} dex")
ax.legend()
plt.tight_layout()

Position noise eats pair counts from the small separations up, which is exactly where the
clustering signal lives. The left panel is the honest way to judge whether a noise level is
reasonable: a condition that removes most of the close pairs is not degrading the signal, it
is deleting it, and everyone scores near zero regardless of what they built.

Now the cost, on the representation that survived every symmetry.

In [ ]:
started = time.time()
corr_x = {cond: [] for cond in CORRUPTIONS}
for suite in PUBLIC_SUITES:
    for sim_id in track(local_split(suite)["test"][:N_TEST_PER_SUITE],
                        f"{suite}: corruptions"):
        c = load(suite, sim_id)
        for cond in CORRUPTIONS:
            moved = apply_condition(c, condition(cond),
                                    condition_seed(BASE_SEED, cond, suite, sim_id))
            corr_x[cond].append(catalog_features(moved))

scaler, model = fits["summary"]
clean = scores[("summary", "clean")]
print(f"{'condition':14s} {'Omega_m':>9s} {'change':>9s} {'sigma_8':>9s} {'change':>9s}")
print(f"{'clean':14s} {clean[0]:9.3f} {'':>9s} {clean[1]:9.3f}")
for cond in CORRUPTIONS:
    prediction = model.predict(scaler.transform(np.asarray(corr_x[cond])))
    values = [r2(test_y[:, j], prediction[:, j]) for j in range(2)]
    print(f"{cond:14s} {values[0]:9.3f} {values[0] - clean[0]:+9.3f} "
          f"{values[1]:9.3f} {values[1] - clean[1]:+9.3f}")
print(f"\n[{time.time() - started:.0f}s]")

Three things to take from those numbers, and the third is the useful one.

**`vel_noise` costs this model exactly nothing** — not because the shift is weak, but because
these 32 features never look at a velocity. The condition is structurally invisible to it. A
condition that cannot move your score is not telling you your model is robust; it is telling
you your model is not reading that column. Notebook 04's baseline does read velocities, and
it does move.

**`mass_noise` barely registers.** A per-object log-normal factor leaves the mass *function*
almost unchanged, and this representation is built out of the mass function.

**`pos_noise_hi` is the interesting one.** It is deliberately near the edge of what is
survivable, and you can see from the left panel above why: at that sigma most pairs inside
1 cMpc/h are gone. If your model does much better than the floor here, you are doing something
that does not depend on small-scale clustering — and that is worth knowing about your own
model.

## 5. The condition you cannot practise on

The last one is a **different simulation code**, held out entirely. You never see it, and no
amount of local validation will tell you how you do on it.

What you can do is build the same shape of test yourself: train on two of your three suites
and score on the third. It is not the same code you will be tested on, but it is the same
kind of gap.

In [ ]:
held_out = {}
for suite in PUBLIC_SUITES:
    train_mask = np.repeat([s != suite for s in PUBLIC_SUITES], N_TRAIN_PER_SUITE)
    test_mask = np.repeat([s == suite for s in PUBLIC_SUITES], N_TEST_PER_SUITE)

    x = np.asarray(train_x["summary"])[train_mask]
    scaler_o = StandardScaler().fit(x)
    model_o = RidgeCV(alphas=np.logspace(-3, 4, 30)).fit(scaler_o.transform(x),
                                                         train_y[train_mask])
    prediction = model_o.predict(
        scaler_o.transform(np.asarray(test_x[("summary", "clean")])[test_mask]))
    held_out[suite] = [r2(test_y[test_mask][:, j], prediction[:, j]) for j in range(2)]

print(f"{'trained on':34s} {'tested on':14s} {'Omega_m':>9s} {'sigma_8':>9s}")
print(f"{'all three':34s} {'all three':14s} {clean[0]:9.3f} {clean[1]:9.3f}")
for suite, values in held_out.items():
    others = " + ".join(s for s in PUBLIC_SUITES if s != suite)
    print(f"{others:34s} {suite:14s} {values[0]:9.3f} {values[1]:9.3f}")

That drop is the event. Nothing was corrupted and no symmetry was applied — the only thing
that changed is which code wrote the catalogs, and the score falls anyway. Closing part of
that gap is the thing worth doing here, and $\sigma_8$ is where it is hardest: it is expected
to fail for most methods, which is why a positive held-out-code $\sigma_8$ has its own prize.

## What you should take away

1. A symmetry is a change that does not alter the physics. If your score moves, that is your
   model, not the data.
2. Which invariance you get is a property of your **representation**, not of your training.
   Sorting rows bought one; dropping absolute coordinates bought two more.
3. A corruption really does remove information. Losing accuracy there is expected; the
   question is how much, relative to the floor.
4. A condition that cannot move your score is telling you which columns you are ignoring.
5. The unseen simulation code is the hard one, and you can build a proxy for it out of the
   data you already have.

Notebook 04 takes the same instruments to a graph neural network.